# Visual Intelligence — End-to-End Colab Demo

This notebook clones the `codex/end-to-end-pipeline` branch, installs dependencies, runs the dependency-free tests, and executes the complete reference-image + video pipeline using the included `test_image.jpg` and `test_video.mp4`. The first run downloads several gigabytes of model weights and may take a few minutes.

Before selecting **Runtime → Run all**, select **Runtime → Change runtime type → GPU**. An L4 or A100 is preferred; the notebook uses Qwen3-VL-2B to keep the smoke test smaller.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU with Runtime → Change runtime type → GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os
import subprocess

repo_dir = Path("/content/Visual-Intelligence")
branch = "codex/end-to-end-pipeline"
repo_url = "https://github.com/SamhithKakarla/Visual-Intelligence.git"

if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "switch", branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)

os.chdir(repo_dir)
print("Working directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["ls", "-lh", "test_image.jpg", "test_video.mp4"], check=True)

## Install dependencies

The first run installs FFmpeg and Python packages. It will also download YOLO, InsightFace, and Qwen weights later during inference.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install --upgrade -q pip setuptools wheel
!python -m pip install -q -r requirements.txt

## Validate the control flow

In [ ]:
!python -m unittest discover -s tests -v

## Run Phase 1 → conditional Phase 2

This smoke-test configuration searches at 2 FPS and limits Phase 2 to 16 target-highlighted evidence frames.

In [ ]:
import json
from pathlib import Path

from visual_intelligence.config import Phase1Config, Phase2Config, PipelineConfig
from visual_intelligence.pipeline import run_pipeline

config = PipelineConfig(
    phase1=Phase1Config(
        search_fps=2.0,
        identity_threshold=0.4,
        max_evidence_frames=16,
    ),
    phase2=Phase2Config(model_id="Qwen/Qwen3-VL-2B-Instruct"),
    runs_dir=Path("/content/visual_intelligence_runs"),
    keep_artifacts=True,
)

result = run_pipeline(
    reference_image="test_image.jpg",
    video_path="test_video.mp4",
    output_path="result.json",
    config=config,
)

print(json.dumps(result.to_dict(), indent=2))

## Inspect the public and diagnostic outputs

In [ ]:
import json
from IPython.display import JSON, display
from PIL import Image

with open("result.json") as stream:
    public_result = json.load(stream)
with open("result.debug.json") as stream:
    debug_result = json.load(stream)

display(JSON(public_result))
display(JSON(debug_result))

if debug_result["phase1"]["person_exists"]:
    evidence_path = debug_result["phase1"]["appearances"][0]["frames"][0]["context_frame_path"]
    print("First target-highlighted evidence frame:", evidence_path)
    display(Image.open(evidence_path))
else:
    print("Phase 1 reported that the reference person was absent, so Phase 2 was correctly skipped.")

## Download the results

Running the next cell packages both JSON outputs into one download.

In [ ]:
from google.colab import files

!zip -j -q visual_intelligence_outputs.zip result.json result.debug.json
files.download("visual_intelligence_outputs.zip")